In [10]:
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import io
import numpy as np
from keras import layers, Model, Input

In [ ]:
def repair_csv_join_lines(path, out_path=None, encoding="utf-8"):
    """
    Считывает файл построчно, склеивает строки пока число " не чётно,
    записывает результат в out_path (если указан) или возвращает содержимое как строку.
    """
    repaired_lines = []
    with open(path, "r", encoding=encoding, errors="replace") as f:
        buffer = ""
        for raw in f:
            line = raw.rstrip("\n")
            if buffer == "":
                buffer = line
            else:
                buffer += "\n" + line  # Сохраняем реальную структуру, если внутри поля есть переводы строки
            # Подсчёт кавычек в buffer
            if buffer.count('"') % 2 == 0:
                repaired_lines.append(buffer)
                buffer = ""
        # Если buffer не пустой после прохода — добавляем как есть 
        if buffer:
            repaired_lines.append(buffer)
    repaired_text = "\n".join(repaired_lines)
    if out_path:
        with open(out_path, "w", encoding=encoding) as fo:
            fo.write(repaired_text)
        return out_path
    return io.StringIO(repaired_text)

files = {
    "test_stances": "test_stances_unlebeledb.csv",
    "test_bodies": "test_bodies.csv",
    "train_bodies": "train_bodies.csv",
    "train_stances": "train_stances.csv"
}

def safe_read_csv(path, index_col='Unnamed: 0', encoding="utf-8"):
    try:
        return pd.read_csv(path, index_col=index_col, encoding=encoding)
    except Exception as e:
        try:
            return pd.read_csv(path, engine="python", index_col=index_col, encoding=encoding)
        except Exception:
            # Восстановление и чтение из буфера
            buff = repair_csv_join_lines(path, encoding=encoding)
            return pd.read_csv(buff, index_col=index_col, encoding=encoding)

dfs = {}
for name, p in files.items():
    dfs[name] = safe_read_csv(p)

test_stances = dfs["test_stances"]
test_bodies = dfs["test_bodies"]
train_bodies = dfs["train_bodies"]
train_stances = dfs["train_stances"]


In [96]:
data = pd.read_csv('data_combined.csv')
data

,Body ID,Headline,Stance,Headline1,Headline2,articleBody,articleBody1,articleBody2
0,1,"['российский', 'бюджет', 'марте', 'недополучил...",agree,российский бюджет в марте недополучил более 30...,российский бюджет март недополучать миллиард р...,Разница между ожидаемыми по итогам марта нефте...,разница ожидаемыми итогам марта нефтегазовыми ...,разница ожидать итог март нефтегазовый доход ф...
1,2,"['банк', 'россии', 'решил', 'снизить', 'ключев...",agree,банк россии решил снизить ключевую ставку с 20...,банк россия решать снижать ключевой ставка,Совет директоров Банка России решил снизить кл...,совет директоров банка россии решил снизить кл...,совет директор банк россия решать снижать ключ...
2,3,"['мыс', 'идокопас', 'нато', 'назвали', 'первую...",disagree,мыс идокопас в нато назвали первую цель для н...,мыс идокопас нато называть первый цель начало ...,В НАТО составили наступательный план действий ...,нато составили наступательный план действий за...,нато составлять наступательный план действие з...
3,4,"['украине', 'прошли', 'празднования', '300', '...",disagree,на украине прошли празднования 300 летия побед...,украина проходить празднование летие победа ка...,Президент Владимир Зеленский принял в Полтаве ...,президент владимир зеленский принял полтаве па...,президент владимир зеленский принимать полтава...
4,5,"['минобороны', 'рф', 'заявило', 'ударах', 'рак...",agree,минобороны рф заявило об ударах ракетами «кали...,минобороны заявлять удар ракета калибр запорож...,"Минобороны России объявило, что ракетами больш...",минобороны россии объявило ракетами большой да...,минобороны россия объявлять ракета большой дал...
...,...,...,...,...,...,...,...,...
4403,4404,"['чиновница', 'призвавшая', '«найти', 'покарат...",disagree,чиновница призвавшая «найти и покарать предате...,чиновница призывать находить покарать предател...,Заместитель министра просвещения Ирина Карепин...,заместитель министра просвещения ирина карепин...,заместитель министр просвещение ирина карепин ...
4404,4405,"['«включим', 'программу', 'киселёва', 'покажем...",disagree,«включим программу киселёва и покажем телевизо...,включать программа киселев показывать телевизо...,Глава штабов Навального Леонид Волков призвал ...,глава штабов навального леонид волков призвал ...,глава штаб навальный леонид волк призывать сво...
4405,4406,"['китайские', 'власти', 'могут', 'казнить', 'з...",disagree,китайские власти могут казнить задержанного по...,китайский власть мочь казнить задержанный подо...,Один из менеджеров стройки космодрома Восточны...,менеджеров стройки космодрома восточный взят с...,менеджер стройка космодром восточный взять стр...
4406,4407,"['луганской', 'области', 'переполнены', 'морги...",agree,в луганской области переполнены морги хранить...,луганский область переполнять морг хранить тел...,Морги Луганской области переполнены телами пог...,морги луганской области переполнены телами пог...,морг луганский область переполнять тело погибш...


In [13]:
for i in tqdm(range(test_stances.shape[0])):
    for j in range(test_bodies.shape[0]):
        if train_bodies.loc[j,'Body ID']==train_stances.loc[i,'Body ID']:
            test_stances.loc[i,'articleBody'] = test_bodies.loc[j,'articleBody']
            test_stances.loc[i,'articleBody1'] = test_bodies.loc[j,'articleBody1']
            test_stances.loc[i,'articleBody2'] = test_bodies.loc[j,'articleBody2']
    test_stances.to_csv('data_test_combined.csv',index=False)

  0%|          | 0/1101 [00:00<?, ?it/s]

In [14]:
data_test = pd.read_csv('data_test_combined.csv')
data_test

,Body ID,label,Headline,Headline1,Headline2,articleBody,articleBody1,articleBody2
0,1,0,"['лукашенко', 'пригрозил', 'литовским', 'танка...",лукашенко пригрозил литовским танкам белорусск...,лукашенко пригрозить литовский танк белорусски...,"Президент Беларуси Александр Лукашенко заявил,...",президент беларуси александр лукашенко заявил ...,президент беларусь александр лукашенко заявлят...
1,2,1,"['российские', 'компании', 'оказались', 'опасн...",российские компании оказались в опасности из з...,российский компания оказываться опасность глоб...,Несколько крупнейших российских компаний оказа...,несколько крупнейших российских компаний оказа...,несколько крупный российский компания оказыват...
2,3,0,"['лукашенко', 'объявил', 'отмене', 'выборов', ...",лукашенко объявил об отмене выборов из за напа...,лукашенко объявлять отмена выборы нападение ин...,Президент Белоруссии Александр Лукашенко объяв...,президент белоруссии александр лукашенко объяв...,президент белоруссия александр лукашенко объяв...
3,4,1,"['«роснефть»', 'стала', 'лидером', 'объему', '...",«роснефть» стала лидером по объему биржевых пр...,роснефть становиться лидер объем биржевой прод...,С начала июля нефтяные компании заметно увелич...,начала июля нефтяные компании заметно увеличил...,начинать июль нефтяной компания заметно увелич...
4,5,1,"['псковской', 'области', 'дадут', 'десятки', '...",псковской области дадут десятки миллионов рубл...,псковский область давать десяток миллион рубль...,Псковская область получит 189 миллионов рублей...,псковская область получит 189 миллионов рублей...,псковский область получать миллион рубль ремон...
...,...,...,...,...,...,...,...,...
1096,1097,0,"['борис', 'джонсон', 'пообещал', 'избирателям'...",борис джонсон пообещал избирателям вернуть ирл...,борис джонсон пообещать избиратель вернуть ирл...,Кандидат в премьер-министры Великобритании Бор...,кандидат премьер министры великобритании борис...,кандидат премьер министр великобритания борис ...
1097,1098,0,"['михаил', 'горбачев', 'заявил', 'подавал', 'о...",михаил горбачев заявил что не подавал в отстав...,михаил горбачев заявлять подавать отставка пос...,Сенсационное заявление сделал 8 декабря в эфир...,сенсационное заявление сделал 8 декабря эфире ...,сенсационный заявление сделать декабрь эфир не...
1098,1099,0,"['мгимо', 'появится', 'факультет', 'международ...",в мгимо появится факультет международных отнош...,мгимо появляться факультет международный отнош...,Московский государственный институт международ...,московский государственный институт международ...,московский государственный институт международ...
1099,1100,0,"['россии', 'введут', 'штраф', 'отрицание', 'аг...",в россии введут штраф за отрицание агрессии нато,россия вводить штраф отрицание агрессия нато,На рассмотрение в Государственную думу поступи...,рассмотрение государственную думу поступил зак...,рассмотрение государственный дума поступать за...


In [98]:
mask = (data.columns != 'Body ID') & (data.columns != 'articleBody') & (data.columns != 'Stance')
data = data.loc[:, ~mask]
data

,Body ID,Stance,articleBody
0,1,agree,Разница между ожидаемыми по итогам марта нефте...
1,2,agree,Совет директоров Банка России решил снизить кл...
2,3,disagree,В НАТО составили наступательный план действий ...
3,4,disagree,Президент Владимир Зеленский принял в Полтаве ...
4,5,agree,"Минобороны России объявило, что ракетами больш..."
...,...,...,...
4403,4404,disagree,Заместитель министра просвещения Ирина Карепин...
4404,4405,disagree,Глава штабов Навального Леонид Волков призвал ...
4405,4406,disagree,Один из менеджеров стройки космодрома Восточны...
4406,4407,agree,Морги Луганской области переполнены телами пог...


In [99]:
data.loc[:, 'Stance'] = data['Stance'].replace(['agree', 'disagree'], [1, 0]).astype('int64', copy=False)
data = data.set_index("Body ID")
data.rename(columns={'articleBody': 'features'}, inplace=True)
data.rename(columns={'Stance': 'label'}, inplace=True)
data.to_csv('data.csv', index='Body ID')
data

/tmp/ipython-input-1669230849.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data.loc[:, 'Stance'] = data['Stance'].replace(['agree', 'disagree'], [1, 0]).astype('int64', copy=False)


,label,features
Body ID,,
1,1,Разница между ожидаемыми по итогам марта нефте...
2,1,Совет директоров Банка России решил снизить кл...
3,0,В НАТО составили наступательный план действий ...
4,0,Президент Владимир Зеленский принял в Полтаве ...
5,1,"Минобороны России объявило, что ракетами больш..."
...,...,...
4404,0,Заместитель министра просвещения Ирина Карепин...
4405,0,Глава штабов Навального Леонид Волков призвал ...
4406,0,Один из менеджеров стройки космодрома Восточны...


In [15]:
mask = (data_test.columns != 'Body ID') & (data_test.columns != 'articleBody') & (data_test.columns != 'label')
data_test = data_test.loc[:, ~mask]
data_test = data_test.set_index("Body ID")
data_test.rename(columns={'articleBody': 'features'}, inplace=True)
data_test.to_csv("data_test.csv", index=False)
data_test

,label,features
Body ID,,
1,0,"Президент Беларуси Александр Лукашенко заявил,..."
2,1,Несколько крупнейших российских компаний оказа...
3,0,Президент Белоруссии Александр Лукашенко объяв...
4,1,С начала июля нефтяные компании заметно увелич...
5,1,Псковская область получит 189 миллионов рублей...
...,...,...
1097,0,Кандидат в премьер-министры Великобритании Бор...
1098,0,Сенсационное заявление сделал 8 декабря в эфир...
1099,0,Московский государственный институт международ...


In [93]:
total_nan=data.isna().sum()
print(f"Общее количество NaN: {total_nan}")

Общее количество NaN: label       0
features    0
dtype: int64


In [94]:
total_nan=data_test.isna().sum()
print(f"Общее количество NaN: {total_nan}")

Общее количество NaN: label       0
features    0
dtype: int64


In [17]:
from keras.layers import TextVectorization
import tensorflow as tf
import re

In [ ]:
MAX_VOCAB = 10000
SEQ_LEN = 500
BATCH = 32
SEED = 42

data = pd.read_csv('data.csv')
data_test = pd.read_csv('data_test.csv')

def clean_text(s):
  s = s.replace("\r", " ").replace("\n", " ")
  s = re.sub(r"\s+", " ", s)
  return s.strip().lower()

data["features"] = data["features"].astype(str).map(clean_text)
data_test["features"] = data_test["features"].astype(str).map(clean_text)

# Метки к int 
data["label"] = data["label"].astype(int)
data_test["label"] = data_test["label"].astype(int)

# Формирование массивов из отдельных файлов (без train_test_split)
x_train_texts = data["features"].values
y_train = data["label"].values
x_test_texts = data_test["features"].values
y_test = data_test["label"].values

# TextVectorization
vectorizer = TextVectorization(
    max_tokens=MAX_VOCAB,
    output_mode="int",
    output_sequence_length=SEQ_LEN,
    standardize=None,
    split="whitespace"
)

# Адаптация только на train-данных
vectorizer.adapt(tf.data.Dataset.from_tensor_slices(x_train_texts).batch(256))

# tf.data pipeline с векторизацией внутри
def make_ds(texts, labels=None, shuffle=False):
    if labels is None:
        ds = tf.data.Dataset.from_tensor_slices(texts)
        ds = ds.map(lambda t: vectorizer(t), num_parallel_calls=tf.data.AUTOTUNE)
    else:
        ds = tf.data.Dataset.from_tensor_slices((texts, labels))
        if shuffle:
            ds = ds.shuffle(buffer_size=4096, seed=SEED)
        ds = ds.map(lambda t, y: (vectorizer(t), tf.cast(y, tf.int32)),
                    num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_ds(x_train_texts, y_train, shuffle=True)
test_ds  = make_ds(x_test_texts, y_test, shuffle=False)

In [118]:
from keras.callbacks import ReduceLROnPlateau, EarlyStopping


callbacks_list = [
    ReduceLROnPlateau(
        monitor='val_acc',
        factor=0.1,
        patience=10,
        mode='min'
    ),

    EarlyStopping(
        monitor='val_acc',
        patience=7,
        restore_best_weights=True,
        mode='max'
    )
]

In [119]:
history = model.fit(
    train_ds,
    epochs=50,
    callbacks=callbacks_list,
    validation_data=test_ds
)

Epoch 1/50
138/138 ━━━━━━━━━━━━━━━━━━━━ 141s 1s/step - accuracy: 0.7016 - loss: 0.5257 - val_accuracy: 0.9110 - val_loss: 0.1953 - learning_rate: 0.0010
Epoch 2/50


/usr/local/lib/python3.12/dist-packages/keras/src/callbacks/callback_list.py:145: UserWarning: Learning rate reduction is conditioned on metric `val_acc` which is not available. Available metrics are: accuracy,loss,val_accuracy,val_loss,learning_rate.
  callback.on_epoch_end(epoch, logs)
/usr/local/lib/python3.12/dist-packages/keras/src/callbacks/early_stopping.py:153: UserWarning: Early stopping conditioned on metric `val_acc` which is not available. Available metrics are: accuracy,loss,val_accuracy,val_loss,learning_rate
  current = self.get_monitor_value(logs)


138/138 ━━━━━━━━━━━━━━━━━━━━ 142s 1s/step - accuracy: 0.9763 - loss: 0.0810 - val_accuracy: 0.9637 - val_loss: 0.1081 - learning_rate: 0.0010
Epoch 3/50
138/138 ━━━━━━━━━━━━━━━━━━━━ 140s 1s/step - accuracy: 0.9942 - loss: 0.0271 - val_accuracy: 0.9491 - val_loss: 0.1717 - learning_rate: 0.0010
Epoch 4/50
138/138 ━━━━━━━━━━━━━━━━━━━━ 154s 1s/step - accuracy: 0.9917 - loss: 0.0272 - val_accuracy: 0.9619 - val_loss: 0.1168 - learning_rate: 0.0010
Epoch 5/50
138/138 ━━━━━━━━━━━━━━━━━━━━ 190s 1s/step - accuracy: 0.9912 - loss: 0.0229 - val_accuracy: 0.9609 - val_loss: 0.1209 - learning_rate: 0.0010
Epoch 6/50
138/138 ━━━━━━━━━━━━━━━━━━━━ 141s 1s/step - accuracy: 0.9966 - loss: 0.0147 - val_accuracy: 0.9609 - val_loss: 0.1608 - learning_rate: 0.0010
Epoch 7/50
138/138 ━━━━━━━━━━━━━━━━━━━━ 139s 1s/step - accuracy: 0.9965 - loss: 0.0099 - val_accuracy: 0.9628 - val_loss: 0.1475 - learning_rate: 0.0010
Epoch 8/50
138/138 ━━━━━━━━━━━━━━━━━━━━ 138s 996ms/step - accuracy: 0.9938 - loss: 0.0223 - v

In [120]:
model.save("newsClassifier.h5")

In [121]:
model.save("newsClassifier.keras")

In [ ]:
# Сохранить vocab как список строк 
vocab = vectorizer.get_vocabulary()
import pickle
with open("vocab.pkl", "wb") as f:
    pickle.dump(vocab, f)

# Или как текстовый файл:
with open("vocab.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(vocab))

In [128]:
model.export("modelParam/newsClassifier")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:965: UserWarning: Layer 'conv1d' (of type Conv1D) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Saved artifact at 'modelParam/newsClassifier'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 500), dtype=tf.int32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  140114801185744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140114801196880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140114801194768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140114748573072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140114748573264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140114801196688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140114748572112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140114801195920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140114748572304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140114748573456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140114748574224

In [26]:
from sklearn.metrics import classification_report
import numpy as np
import tensorflow as tf

# Загружаем слой
layer = tf.keras.layers.TFSMLayer("newsClassifier/", call_endpoint='serving_default')

# Оборачиваем в Keras Model, чтобы пользоваться .predict
inputs = tf.keras.Input(shape=(SEQ_LEN,), dtype=tf.int32)
outputs = layer(inputs)
model = tf.keras.Model(inputs, outputs)

# Собираем тестовые данные
x_test_list, y_test_list = [], []
for x_batch, y_batch in test_ds:
    x_test_list.append(x_batch.numpy())
    y_test_list.append(y_batch.numpy())

x_test = np.concatenate(x_test_list, axis=0)
y_test = np.concatenate(y_test_list, axis=0)

print(x_test.shape, y_test.shape)

# Предсказания
y_prob = model.predict(x_test, batch_size=32)

# Если predict возвращает dict, достаём массив
if isinstance(y_prob, dict):
    y_prob = list(y_prob.values())[0]

y_pred = (y_prob >= 0.5).astype(int).ravel()

from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred, target_names=["Fake", "Real"]))



(1101, 500) (1101,)
35/35 ━━━━━━━━━━━━━━━━━━━━ 5s 141ms/step
              precision    recall  f1-score   support

        Fake       0.99      0.92      0.95       531
        Real       0.93      0.99      0.96       570

    accuracy                           0.96      1101
   macro avg       0.96      0.96      0.96      1101
weighted avg       0.96      0.96      0.96      1101

